# 실습 3 · Prompt Engineering + JSON 구조화

네트워크 연결 한 줄을 JSON 판정으로 바꾸고 정답과 비교합니다.
- 공격 유형은 실습 데이터에 있는 DoS · Exploits · Reconnaissance 3가지와 Normal(정상) 중에서만 고르게 합니다.
- LLM에는 정답을 넣지 않습니다. 정답은 `../data/sample_logs_truth.csv`에서 따로 읽어 비교에만 씁니다.
- JSON이 깨지면 한 번 다시 묻고, 그래도 안 되면 '정상'이 아니라 판정불가(사람 검토) 로 둡니다.

In [2]:
from openai import OpenAI

ENV = "mono"          # "ollama" = 내 PC,  "mono" = MonoRouter(원격)
if ENV == "mono":
    client = OpenAI(base_url="https://monogpt.kr/api/monorouter/v1/openai/v1",
                    api_key="mr_openai_e0di9bdw3iFdUDG1hgAxEfo_tH_DXI2z")
    MODEL = "gpt-5.5"
else:
    client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    MODEL = "llama3.2:3b"
print(ENV, MODEL)

mono gpt-5.5


In [4]:
def ask(prompt, system=None, temperature=0.2, show=True):
    """질문을 보내고 답을 받아 돌려줌. show=True면 답이 만들어지는 대로 화면에 표시"""
    messages = [{"role": "user", "content": prompt}]
    if system:
        messages.insert(0, {"role": "system", "content": system})
    limit = {"max_tokens": 500} if ENV == "ollama" else {}   # 작은 모델이 끝없이 길게 답하는 것을 막음
    stream = client.chat.completions.create(model=MODEL, messages=messages,
                                            temperature=temperature, stream=True, **limit)
    answer = ""
    for chunk in stream:
        piece = chunk.choices[0].delta.content if chunk.choices else None
        if piece:
            answer += piece
            if show:
                print(piece, end="", flush=True)
    if show:
        print()
    return answer

In [5]:
import json, re

def parse_json(raw):
    """답에서 { ... } 부분만 꺼내 JSON으로 읽음. 없거나 깨졌으면 None"""
    m = re.search(r"\{.*\}", raw or "", re.DOTALL)
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None

## 프롬프트

In [6]:
TYPES = ["DoS", "Exploits", "Reconnaissance", "Normal"]

PROMPT = """당신은 금융기관 SOC 분석가입니다.
다음 네트워크 연결 기록을 보고 JSON으로만 답하세요.

필드 뜻: src/dst=출발지/목적지 IP:포트, service=서비스(-는 식별 안 됨), state=연결 상태(FIN 정상 종료, INT 응답 없음, CON 연결 중),
dur=지속시간(초), spkts/dpkts=송신/수신 패킷 수, sbytes/dbytes=송신/수신 바이트, rate=초당 패킷 수,
ct_dst_ltm=최근 100개 연결 중 같은 목적지 연결 수, ct_src_dport_ltm=최근 100개 연결 중 같은 출발지·같은 목적지 포트 연결 수

공격 유형 (이 4개 중 하나만 고름):
- DoS: 서비스를 쓰지 못하게 만드는 공격
- Exploits: 알려진 취약점을 이용한 침투
- Reconnaissance: 포트 · 서비스를 탐색하는 정보 수집
- Normal: 정상 연결

연결 기록:
{flow}

출력 형식:
{{"attack_type": "DoS / Exploits / Reconnaissance / Normal 중 하나",
  "risk_score": 0~100 정수,
  "reason": "판단에 쓴 필드와 값을 넣은 한 줄 설명"}}"""

def analyze(flow):
    for _ in range(2):                                        # 형식이 틀리면 한 번 더
        d = parse_json(ask(PROMPT.format(flow=flow), system="JSON만 출력하는 보안 분석가."))
        if d and d.get("attack_type") in TYPES:
            return d
    return {"attack_type": "판정불가"}                          # 정상으로 처리하지 않음 → 사람 검토

## 샘플 4건과 정답 (sample_logs.txt 8 · 12 · 1 · 2번째 줄)

In [9]:
import pandas as pd

LINES = open("../data/sample_logs.txt", encoding="utf-8").read().splitlines()
truth = pd.read_csv("../data/sample_logs_truth.csv").set_index("line")["attack_cat"]
PICK = [8, 12, 1, 2]            #로그에서 특정 줄의 로그를 선택하기 위한 번호,  4개 로그만 이용함
for n in PICK:
    print(n, LINES[n - 1][28:120])

8 src=175.45.176.1:20549 dst=149.171.126.10:111 proto=udp service=- state=INT dur=0.000 spkts=
12 src=175.45.176.1:63277 dst=149.171.126.18:110 proto=tcp service=pop3 state=FIN dur=1.272 spk
1 src=175.45.176.1:12940 dst=149.171.126.16:80 proto=tcp service=http state=FIN dur=0.268 spkt
2 src=59.166.0.3:46973 dst=149.171.126.0:80 proto=tcp service=http state=FIN dur=9.108 spkts=8


## 분석과 비교

In [10]:
rows = []
for n in PICK:
    flow = LINES[n - 1][28:]                       # 앞의 날짜 · [FLOW] 표시는 뺌
    print(f"\n[{n}번째 줄]")
    result = analyze(flow)
    rows.append({"줄": n, "예측": result["attack_type"], "정답": truth[n],
                 "일치": result["attack_type"] == truth[n]})

table = pd.DataFrame(rows)
print(table.to_string(index=False))
print(f"\n일치 {table['일치'].sum()}/{len(table)}, 판정불가 {(table['예측'] == '판정불가').sum()}건")


[8번째 줄]
{"attack_type":"Reconnaissance","risk_score":55,"reason":"udp 목적지 포트 111로 service=-, state=INT, dpkts=0 응답 없음이며 spkts=2/sbytes=168의 소량 단발 접근(ct_dst_ltm=1, ct_src_dport_ltm=1)으로 DoS보다는 RPC 서비스 탐색 시도로 판단"}

[12번째 줄]
{"attack_type":"Normal","risk_score":15,"reason":"pop3 서비스(tcp/110) 연결이 state=FIN으로 정상 종료되었고 dur=1.272, spkts/dpkts=34/116, ct_dst_ltm=1, ct_src_dport_ltm=1로 반복 접속·스캔·DoS 징후가 낮음"}

[1번째 줄]
{"attack_type":"Normal","risk_score":10,"reason":"service=http, state=FIN 정상 종료, dur=0.268초, spkts/dpkts=10/8 및 sbytes/dbytes=878/1002로 양방향 응답이 정상적이며 ct_dst_ltm=1, ct_src_dport_ltm=1로 반복/스캔 징후가 낮음"}

[2번째 줄]
{"attack_type":"Normal","risk_score":5,"reason":"tcp http 80 연결이 state=FIN으로 정상 종료되었고 dur=9.108, spkts=8/dpkts=10, rate=2.0, ct_dst_ltm=4, ct_src_dport_ltm=1로 과도한 반복·스캔·DoS 징후가 낮음"}
 줄             예측             정답    일치
 8 Reconnaissance Reconnaissance  True
12         Normal       Exploits False
 1         Normal            DoS False
 2         Normal         Normal  True

일치